## 1. Introduction

## 2. Data Acquisition

In [10]:
import requests
import pandas as pd
import json
import time
import os

In [11]:
with open("../keys.json") as f:
    keys = json.load(f)

API_KEYS = [
    keys["guardian"]["api_key_1"],
    keys["guardian"]["api_key_2"],
    keys["guardian"]["api_key_3"]
]


In [12]:
acled = pd.read_csv("../data/cleaned_fatalities.csv", index_col=0)

print("=== ACLED Dataset ===")
print(f"Shape: {acled.shape}")
print(f"Columns: {list(acled.columns)}")
print(f"Countries: {acled['COUNTRY'].nunique()} unique")
print(f"Year range: {acled['YEAR'].min()} – {acled['YEAR'].max()}")
acled.head()

=== ACLED Dataset ===
Shape: (2943, 3)
Columns: ['COUNTRY', 'YEAR', 'FATALITIES']
Countries: 245 unique
Year range: 1997 – 2026


,COUNTRY,YEAR,FATALITIES
1,Afghanistan,2017,36360
2,Afghanistan,2018,42991
3,Afghanistan,2019,41419
4,Afghanistan,2020,31037
5,Afghanistan,2021,42425


In [ ]:
FROM_DATE = "2015-01-01"
TO_DATE   = "2024-12-31"

# --- Automatic conversion: lowercase and replace spaces with hyphens ---
acled_countries = acled["COUNTRY"].unique().tolist()

def name_to_slug(name):
    return name.lower().strip().replace(" ", "-").replace("'", "").replace(",", "")

auto_slugs = {country: name_to_slug(country) for country in acled_countries}

{'Afghanistan': 'afghanistan',
 'Akrotiri and Dhekelia': 'akrotiri-and-dhekelia',
 'Albania': 'albania',
 'Algeria': 'algeria',
 'American Samoa': 'american-samoa',
 'Andorra': 'andorra',
 'Angola': 'angola',
 'Anguilla': 'anguilla',
 'Antarctica': 'antarctica',
 'Antigua and Barbuda': 'antigua-and-barbuda',
 'Argentina': 'argentina',
 'Armenia': 'armenia',
 'Aruba': 'aruba',
 'Australia': 'australia',
 'Austria': 'austria',
 'Azerbaijan': 'azerbaijan',
 'Bahamas': 'bahamas',
 'Bahrain': 'bahrain',
 'Bailiwick of Guernsey': 'bailiwick-of-guernsey',
 'Bailiwick of Jersey': 'bailiwick-of-jersey',
 'Bangladesh': 'bangladesh',
 'Barbados': 'barbados',
 'Belarus': 'belarus',
 'Belgium': 'belgium',
 'Belize': 'belize',
 'Benin': 'benin',
 'Bermuda': 'bermuda',
 'Bhutan': 'bhutan',
 'Bolivia': 'bolivia',
 'Bosnia and Herzegovina': 'bosnia-and-herzegovina',
 'Botswana': 'botswana',
 'Brazil': 'brazil',
 'British Indian Ocean Territory': 'british-indian-ocean-territory',
 'British Virgin Island

In [18]:
SLUG_CORRECTIONS = {
    "Democratic Republic of Congo": "democratic-republic-of-congo",
    "Republic of Congo":            "republic-of-congo",
    "United States":                "us-news",
    "Myanmar":                      "myanmar",
    "Ivory Coast":                  "ivory-coast",
    "Burkina Faso":                 "burkina-faso",
    "South Sudan":                  "south-sudan",
    "Central African Republic":     "central-african-republic",
    "Bosnia-Herzegovina":           "bosnia-and-herzegovina",
    "North Korea":                  "north-korea",
    "South Korea":                  "south-korea",
    "Saudi Arabia":                 "saudi-arabia",
    "Sri Lanka":                    "sri-lanka",
    "Papua New Guinea":             "papua-new-guinea",
}

In [19]:
COUNTRY_TO_SLUG = {**auto_slugs, **SLUG_CORRECTIONS}

# Final list of (country_name, guardian_slug) pairs
COUNTRY_LIST = [(country, COUNTRY_TO_SLUG[country]) for country in acled_countries]

print(f"\nTotal countries to query: {len(COUNTRY_LIST)}")
print("\nSample mappings:")
for name, slug in COUNTRY_LIST[:5]:
    print(f"  {name} -> world/{slug}")


Total countries to query: 245

Sample mappings:
  Afghanistan -> world/afghanistan
  Akrotiri and Dhekelia -> world/akrotiri-and-dhekelia
  Albania -> world/albania
  Algeria -> world/algeria
  American Samoa -> world/american-samoa


In [ ]:
BASE_URL    = "https://content.guardianapis.com/search"
current_key = 0  

def fetch_articles(country_name, slug):
    """
    Fetches all Guardian articles tagged with a given country slug
    between FROM_DATE and TO_DATE. Paginates automatically and
    rotates API keys on rate limiting.

    Parameters:
        country_name : str — original ACLED country name, stored in output
        slug         : str — Guardian tag slug for that country

    Returns a list of article dicts.
    """
    global current_key
    articles = []
    page = 1

    while True:
        params = {
            "tag":         f"world/{slug}",
            "from-date":   FROM_DATE,
            "to-date":     TO_DATE,
            "show-fields": "wordcount,sectionName",
            "page-size":   200,
            "page":        page,
            "api-key":     API_KEYS[current_key]
        }

        r = requests.get(BASE_URL, params=params)
        time.sleep(0.1)

        # Rotate key if rate limited
        if r.status_code == 429:
            current_key = (current_key + 1) % len(API_KEYS)
            print(f"  Rate limited — switching to key {current_key}")
            continue

        if r.status_code != 200:
            print(f"  Error {r.status_code} for {slug} page {page} — skipping")
            break

        data    = r.json()["response"]
        results = data.get("results", [])

        if not results:
            break

        for a in results:
            articles.append({
                "country":   country_name,
                "date":      a["webPublicationDate"],
                "section":   a["sectionName"],
                "wordcount": a.get("fields", {}).get("wordcount", None),
                "title":     a["webTitle"],
                "pillar":    a.get("pillarName", None)
            })

        if page >= data["pages"]:
            break

        page += 1

    return articles


In [ ]:
# --- Run collection (skips if file already exists) ---
RAW_PATH = "../data/guardian_raw.csv"

if os.path.exists(RAW_PATH) and os.path.getsize(RAW_PATH) > 0:
    print("Guardian raw data already collected — loading from file.")
    guardian_raw = pd.read_csv(RAW_PATH)
else:
    all_articles = []
    for country_name, slug in COUNTRY_LIST:
        print(f"Fetching: {country_name} (world/{slug})...")
        articles = fetch_articles(country_name, slug)
        all_articles.extend(articles)
        print(f"  -> {len(articles)} articles")

    guardian_raw = pd.DataFrame(all_articles)
    guardian_raw.to_csv(RAW_PATH, index=False)
    print(f"\nSaved {len(guardian_raw)} total articles to {RAW_PATH}")

print(f"\nShape: {guardian_raw.shape}")
print(f"Countries with articles: {guardian_raw['country'].nunique()}")
print(f"Date range: {guardian_raw['date'].min()} to {guardian_raw['date'].max()}")
guardian_raw.head()

## World Bank Income Classification

In [ ]:
OWID_URL = (
    "https://ourworldindata.org/grapher/world-bank-income-groups.csv"
    "?v=1&csvType=full&useColumnShortNames=false"
)

wb_raw = pd.read_csv(
    OWID_URL,
    storage_options={"User-Agent": "Our World In Data data fetch/1.0"}
)

print("=== World Bank Income Classification ===")
print(f"Shape: {wb_raw.shape}")
print(f"Columns: {list(wb_raw.columns)}")
print(f"Years available: {wb_raw['Year'].min()} – {wb_raw['Year'].max()}")

# Keep only the most recent year — we use a single static classification
# per country rather than a time-varying one, which is sufficient for
# our binary Global South / Global North label.
wb_latest = (
    wb_raw
    .sort_values("Year", ascending=False)
    .drop_duplicates(subset="Entity")
    .rename(columns={"Entity": "country", "Year": "wb_year"})
)

# Save locally so the notebook is reproducible without an internet connection
wb_latest.to_csv("../data/worldbank_income.csv", index=False)
print(f"\nSaved to data/worldbank_income.csv")
print(f"Countries classified: {wb_latest['country'].nunique()}")
wb_latest.head()


=== World Bank Income Classification ===
Shape: (7953, 4)
Columns: ['Entity', 'Code', 'Year', "World Bank's income classification"]
Years available: 1987 – 2024

Saved to data/worldbank_income.csv
Countries classified: 226


,country,Code,wb_year,World Bank's income classification
4148,Liberia,LBR,2024,Low-income countries
7914,Zambia,ZMB,2024,Lower-middle-income countries
37,Afghanistan,AFG,2024,Low-income countries
4110,Lesotho,LSO,2024,Lower-middle-income countries
4186,Libya,LBY,2024,Upper-middle-income countries


In [27]:
INCOME_COL = [c for c in wb_latest.columns if "income" in c.lower()][0]
print(f"Using income column: '{INCOME_COL}'")

# Apply binary label
wb_latest["region_group"] = wb_latest[INCOME_COL].apply(
    lambda x: "Global North" if str(x).strip() == "High-income countries" else "Global South"
)

# --- Manual overrides for edge cases ---
# Some countries are misclassified by income group alone for our purposes.
# We document each override explicitly with a reason.
OVERRIDES = {
    # Geopolitically Global North despite upper-middle income classification
    "Ukraine":      "Global North",
    # High income but geopolitically and geographically Global South
    "Saudi Arabia": "Global South",
    "Qatar":        "Global South",
    "Kuwait":       "Global South",
    "Bahrain":      "Global South",
    "Oman":         "Global South",
    "United Arab Emirates": "Global South",
}

for country, label in OVERRIDES.items():
    mask = wb_latest["country"] == country
    if mask.any():
        wb_latest.loc[mask, "region_group"] = label
        print(f"  Override: {country} -> {label}")

# --- Build the two lists ---
# We classify all World Bank countries here without filtering to ACLED yet.
# Name harmonisation between World Bank and ACLED happens in Section 3.
# Filtering to conflict-affected countries only will be applied post-merge.

global_north_countries = sorted(
    wb_latest[wb_latest["region_group"] == "Global North"]["country"].tolist()
)

global_south_countries = sorted(
    wb_latest[wb_latest["region_group"] == "Global South"]["country"].tolist()
)

print(f"Global North countries in ACLED:      {len(global_north_countries)}")
print(f"Global South countries in ACLED: {len(global_south_countries)}")
print(f"Global North sample:      {global_north_countries[:5]}")
print(f"Global South sample: {global_south_countries[:5]}")

# Save the classification back to CSV for use in Section 3
wb_latest.to_csv("../data/worldbank_income.csv", index=False)
print("\nUpdated worldbank_income.csv with region_group column")

Using income column: 'World Bank's income classification'
  Override: Ukraine -> Global North
  Override: Saudi Arabia -> Global South
  Override: Qatar -> Global South
  Override: Kuwait -> Global South
  Override: Bahrain -> Global South
  Override: Oman -> Global South
  Override: United Arab Emirates -> Global South
Global North countries in ACLED:      85
Global South countries in ACLED: 141
Global North sample:      ['American Samoa', 'Andorra', 'Antigua and Barbuda', 'Aruba', 'Australia']
Global South sample: ['Afghanistan', 'Albania', 'Algeria', 'Angola', 'Argentina']

Updated worldbank_income.csv with region_group column


## 3. Preparation of the Data for Analysis

## 4. Exploratory Data Analysis

## 5. Conclusion